# Análisis de Series de tiempo



---

## 📚 Sobre este Material

Este material ha sido diseñado con el propósito de capacitar, actualizar y fortalecer las competencias en el manejo de datos con Python.

### 🤝 Compartir y Colaborar

Este contenido es **libre para compartir, revisar, divulgar y mejorar**. Se promueve activamente su distribución en la comunidad para que más personas puedan beneficiarse y contribuir a su mejora continua. Tu feedback y sugerencias son siempre bienvenidos.

### 👨‍💻 Autor

**Andrés Muñoz**  
*AI & Data Strategy Leader passionate about NLP, LLMs, and MLOps. Driving innovation with data*

- 💼 LinkedIn: [in/amms1989](https://linkedin.com/in/amms1989)
- 🐙 GitHub: [https://github.com/anguihero](https://github.com/anguihero)

---


---------------
## Tabla de contenido
---------------

* 

---
## Modulos

---

In [ ]:
import warnings                          # Supresión de advertencias durante ejecución

# === Manipulación y Análisis de Datos ===
import numpy as np                       # Operaciones numéricas y arrays multidimensionales
import pandas as pd                      # Estructuras de datos (DataFrame, Series) y análisis tabular
import seaborn as sns                    # Visualización estadística de alto nivel (basada en matplotlib)

# === Visualización Interactiva ===
import plotly.graph_objects as go        # Gráficos interactivos personalizados (trazas, scatter, etc.)
from plotly.subplots import make_subplots  # Creación de subplots en figuras Plotly

# === Descomposición y Diagnóstico de Series de Tiempo (Statsmodels) ===
from statsmodels.tsa.seasonal import seasonal_decompose  # Descomposición STL: Tendencia + Estacionalidad + Residuo
from statsmodels.tsa.statespace.sarimax import SARIMAX   # Modelo SARIMA con componentes estacionales y exógenos
from statsmodels.tsa.stattools import adfuller           # Test Augmented Dickey-Fuller (estacionariedad)
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf  # Correlogramas ACF y PACF (identificación de órdenes p,q)
from statsmodels.stats.diagnostic import acorr_ljungbox  # Test Ljung-Box: autocorrelación en residuos del modelo

# === Modelo de Pronóstico Automatizado ===
from prophet import Prophet              # Facebook Prophet: pronóstico con tendencia + estacionalidades múltiples + changepoints

# === Machine Learning ===
from xgboost import XGBRegressor         # XGBoost: regresión con gradient boosting (requiere feature engineering de lags)
from sklearn.preprocessing import MinMaxScaler           # Normalización de series al rango [0,1] (requerido por LSTM)
from sklearn.metrics import mean_absolute_error, mean_squared_error  # Métricas de evaluación: MAE y MSE/RMSE

# === Deep Learning (TensorFlow / Keras) ===
import tensorflow as tf                              # Framework de deep learning (backend de Keras)
from tensorflow.keras.models import Sequential       # Arquitectura secuencial de capas para LSTM
from tensorflow.keras.layers import LSTM, Dense, Dropout  # Capas: LSTM (memoria), Dense (salida), Dropout (regularización)
from tensorflow.keras.callbacks import EarlyStopping # Detiene entrenamiento si val_loss no mejora (evita sobreajuste)

# === Estadística Científica ===
from scipy import stats                  # Distribuciones, tests de normalidad (Jarque-Bera, Shapiro, Q-Q plot)

# === Visualización Estática ===
import matplotlib.pyplot as plt          # Gráficos estáticos base: series, histogramas, subplots diagnósticos

---
## Configuraciones

---

In [ ]:

# === Configuración de Estilos ===
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
COLORS = {
    'real': '#2E86AB',
    'sarima': '#A23B72',
    'prophet': '#F18F01',
    'xgboost': '#C73E1D',
    'lstm': '#3B1F2B',
    'train': '#90E0EF',
    'test': '#F4A261'
}



print("✅ Todas las librerías importadas correctamente")
print(f"   → TensorFlow versión: {tf.__version__}")
print(f"   → Numpy versión: {np.__version__}")
print(f"   → Pandas versión: {pd.__version__}")

In [ ]:

warnings.filterwarnings('ignore')

## Introducción


### 🔬 Laboratorio de Series de Tiempo
#### ¿Qué vamos a aprender?

Este notebook es un laboratorio práctico que cubre el ciclo completo de
análisis y pronóstico de series de tiempo:

1. **Generación de Datos Sintéticos** con componentes realistas
2. **EDA y Diagnóstico** para entender la naturaleza de la serie
3. **Modelos Comparativos**: SARIMA, Prophet, XGBoost y LSTM
4. **Evaluación Rigurosa** con métricas estándar de la industria
5. **Análisis de Residuos** para validar supuestos del modelo ganador

### GENERACIÓN DE DATOS SINTÉTICOS COMPLEJOS


TEORÍA: Una serie de tiempo realista se compone de:
  - Tendencia (T): Dirección de largo plazo
  - Estacionalidad (S): Patrones cíclicos repetitivos (MULTIPLICATIVA: S * T)
  - Cambio Estructural: Perturbaciones externas (e.g., crisis, nuevas políticas)
  - Ruido (ε): Variación aleatoria no explicada ~ N(0, σ²)

Modelo: Y(t) = T(t) * S(t) + Ruptura(t) + ε(t)

In [ ]:


np.random.seed(42)

# === Parámetros del Dataset ===
n_periodos = 365 * 3          # 3 años de datos diarios
fecha_inicio = '2021-01-01'
frecuencia = 'D'               # Frecuencia diaria

# === Índice de Tiempo ===
fechas = pd.date_range(start=fecha_inicio, periods=n_periodos, freq=frecuencia)
t = np.arange(n_periodos)

# === Componente 1: Tendencia Lineal ===
# Representa crecimiento sostenido (ej: ventas de una empresa en expansión)
tendencia = 100 + 0.08 * t

# === Componente 2: Estacionalidad Multiplicativa ===
# Ciclo anual (período=365) → Simula comportamiento estacional (verano/invierno)
estacionalidad_anual = 1 + 0.25 * np.sin(2 * np.pi * t / 365)
# Ciclo semanal (período=7) → Simula picos de fin de semana
estacionalidad_semanal = 1 + 0.10 * np.sin(2 * np.pi * t / 7)
estacionalidad = estacionalidad_anual * estacionalidad_semanal

# === Componente 3: Cambio Estructural (Ruptura) ===
# Simula un evento externo (ej: pandemia, lanzamiento de producto)
punto_ruptura = int(n_periodos * 0.55)   # La ruptura ocurre al 55% de la serie
magnitud_ruptura = 30
ruptura = np.zeros(n_periodos)
ruptura[punto_ruptura:] = magnitud_ruptura

# === Componente 4: Ruido Gaussiano ===
# ε ~ N(0, σ²) → Variabilidad no explicada por el modelo
ruido = np.random.normal(0, 8, n_periodos)

# === Serie Final Combinada ===
# Modelo aditivo-multiplicativo híbrido
serie_valores = tendencia * estacionalidad + ruptura + ruido

# === Construcción del DataFrame ===
df = pd.DataFrame({
    'fecha': fechas,
    'valor': serie_valores,
    'tendencia': tendencia,
    'estacionalidad': estacionalidad,
    'ruptura': ruptura,
    'ruido': ruido
})
df.set_index('fecha', inplace=True)

print("✅ Dataset sintético generado exitosamente")
print(f"   → Período: {df.index[0].date()} a {df.index[-1].date()}")
print(f"   → Observaciones: {len(df):,}")
print(f"   → Punto de ruptura: {df.index[punto_ruptura].date()}")
print(f"\n📊 Estadísticas Descriptivas:")
print(df['valor'].describe().round(2))

### VISUALIZACIÓN DE COMPONENTES

In [ ]:
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=[
        '📈 Serie de Tiempo Completa (con Ruptura Estructural)',
        '📉 Componente: Tendencia',
        '🌊 Componente: Estacionalidad',
        '🔀 Componente: Ruido Gaussiano'
    ],
    vertical_spacing=0.08
)

# Serie completa
fig.add_trace(go.Scatter(
    x=df.index, y=df['valor'],
    mode='lines', name='Valor Real',
    line=dict(color=COLORS['real'], width=1.5)
), row=1, col=1)

# Línea de ruptura
fig.add_vline(
    x=df.index[punto_ruptura],  # Mantén el objeto Timestamp original
    line_dash="dash", 
    line_color="red",
    annotation_text="🔴 Ruptura Estructural",
    annotation_position="top left", # Posición fija para evitar cálculos de promedio
    row=1, col=1
)

# Tendencia
fig.add_trace(go.Scatter(
    x=df.index, y=df['tendencia'],
    mode='lines', name='Tendencia',
    line=dict(color=COLORS['prophet'], width=2)
), row=2, col=1)

# Estacionalidad
fig.add_trace(go.Scatter(
    x=df.index, y=df['estacionalidad'],
    mode='lines', name='Estacionalidad',
    line=dict(color=COLORS['xgboost'], width=1.5)
), row=3, col=1)

# Ruido
fig.add_trace(go.Scatter(
    x=df.index, y=df['ruido'],
    mode='lines', name='Ruido',
    line=dict(color='gray', width=1),
    opacity=0.7
), row=4, col=1)

fig.update_layout(
    height=900,
    title_text='🔬 <b>Descomposición Teórica de la Serie Sintética</b>',
    title_font_size=16,
    showlegend=True,
    template='plotly_white'
)

fig.show()
print("✅ Gráfico de componentes generado")


## EDA

### 📊 Análisis Exploratorio (EDA) y Diagnóstico

#### ¿Por qué es crítico el EDA en Series de Tiempo?
Antes de ajustar cualquier modelo, debemos diagnosticar:
- **¿Es estacionaria la serie?** → La mayoría de modelos ARIMA lo requieren
- **¿Qué tipo de estacionalidad tiene?** → Determina el período S en SARIMA
- **¿Qué órdenes p y q usar?** → Los gráficos ACF/PACF nos lo indican


### DESCOMPOSICIÓN ESTADÍSTICA CON STATSMODELS


TEORÍA: 

seasonal_decompose separa la serie en T + S + R usando medias móviles. El modelo multiplicativo es apropiado cuando la amplitud estacional crece con la tendencia.



In [ ]:


# Usaremos frecuencia semanal (período=7) para la descomposición
# ya que los datos son diarios y tenemos estacionalidad semanal
descomposicion = seasonal_decompose(
    df['valor'],
    model='multiplicative',   # Modelo multiplicativo: Y = T * S * R
    period=7,                  # Período semanal
    extrapolate_trend='freq'
)

# === Visualización con Matplotlib ===
fig, axes = plt.subplots(4, 1, figsize=(14, 10))
fig.suptitle('Descomposición STL de la Serie (Multiplicativa, período=7)',
             fontsize=14, fontweight='bold', y=1.01)

componentes = {
    'Serie Original': (df['valor'], COLORS['real']),
    'Tendencia': (descomposicion.trend, COLORS['prophet']),
    'Estacionalidad': (descomposicion.seasonal, COLORS['xgboost']),
    'Residuo': (descomposicion.resid, 'gray')
}

for ax, (nombre, (datos, color)) in zip(axes, componentes.items()):
    ax.plot(datos, color=color, linewidth=1.2, alpha=0.9)
    ax.set_title(nombre, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.grid(True, alpha=0.3)
    if nombre == 'Tendencia':
        ax.axvline(x=df.index[punto_ruptura], color='red',
                   linestyle='--', alpha=0.7, label='Ruptura')
        ax.legend()

plt.tight_layout()
plt.savefig('descomposicion_serie.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Descomposición completada")


### TEST DE ESTACIONARIEDAD - DICKEY-FULLER AUMENTADO (ADF)

TEORÍA: 
El Test ADF prueba la hipótesis nula de que la serie tiene una raíz unitaria (no estacionaria).
  H₀: La serie tiene raíz unitaria → NO estacionaria
  H₁: La serie NO tiene raíz unitaria → Estacionaria

Si p-value < 0.05 → Rechazamos H₀ → Serie estacionaria ✅

In [ ]:

def test_adf_completo(serie, nombre_serie="Serie"):
    """
    Ejecuta el Test Augmented Dickey-Fuller y presenta resultados
    de forma interpretable.
    """
    resultado = adfuller(serie.dropna(), autolag='AIC')

    print(f"\n{'='*55}")
    print(f"  📋 Test ADF - {nombre_serie}")
    print(f"{'='*55}")
    print(f"  Estadístico ADF : {resultado[0]:.6f}")
    print(f"  p-value         : {resultado[1]:.6f}")
    print(f"  Lags utilizados : {resultado[2]}")
    print(f"  Observaciones   : {resultado[3]}")
    print(f"\n  Valores Críticos:")
    for nivel, valor in resultado[4].items():
        signo = "✅" if resultado[0] < valor else "❌"
        print(f"    {signo} {nivel}: {valor:.6f}")

    if resultado[1] < 0.05:
        print(f"\n  🟢 CONCLUSIÓN: p-value={resultado[1]:.4f} < 0.05")
        print(f"     → SERIE ESTACIONARIA (Rechazamos H₀)")
    else:
        print(f"\n  🔴 CONCLUSIÓN: p-value={resultado[1]:.4f} >= 0.05")
        print(f"     → SERIE NO ESTACIONARIA (No podemos rechazar H₀)")
        print(f"     → Aplicar diferenciación (d=1 o d=2)")

    return resultado

# Test en serie original
resultado_original = test_adf_completo(df['valor'], "Serie Original")

# Diferenciación de primer orden
df['valor_diff1'] = df['valor'].diff()
resultado_diff1 = test_adf_completo(df['valor_diff1'].dropna(),
                                     "Serie Diferenciada (d=1)")

# === Visualización: Media y Varianza Móvil ===
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Análisis de Estacionariedad: Media y Varianza Móvil',
             fontsize=13, fontweight='bold')

for idx, (serie_nombre, serie_datos) in enumerate([
    ('Original', df['valor']),
    ('Diferenciada (d=1)', df['valor_diff1'].dropna())
]):
    ax1 = axes[idx, 0]
    ax2 = axes[idx, 1]

    ventana = 30
    media_movil = serie_datos.rolling(window=ventana).mean()
    std_movil = serie_datos.rolling(window=ventana).std()

    # Media Móvil
    ax1.plot(serie_datos, alpha=0.6, color=COLORS['real'],
             label='Serie', linewidth=0.8)
    ax1.plot(media_movil, color=COLORS['xgboost'],
             label=f'Media Móvil ({ventana}d)', linewidth=2)
    ax1.set_title(f'{serie_nombre} - Media Móvil')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    # Varianza Móvil
    ax2.plot(std_movil, color=COLORS['sarima'],
             label=f'Std Móvil ({ventana}d)', linewidth=2)
    ax2.set_title(f'{serie_nombre} - Desv. Estándar Móvil')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('estacionariedad.png', dpi=150, bbox_inches='tight')
plt.show()


### GRÁFICOS ACF Y PACF

TEORÍA: 
Los correlogramas ACF y PACF son la herramienta diagnóstica principal para identificar los órdenes del modelo ARIMA(p,d,q):

  ACF (Autocorrelación):
    - Corte abrupto en lag q → Proceso MA(q)
    - Decaimiento gradual → Proceso AR o ARMA

  PACF (Autocorrelación Parcial):
    - Corte abrupto en lag p → Proceso AR(p)
    - Decaimiento gradual → Proceso MA o ARMA

Para SARIMA, también buscamos picos en los lags estacionales (7, 14, 21...)


In [ ]:

serie_estacionaria = df['valor_diff1'].dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Correlogramas ACF y PACF (Serie Diferenciada)',
             fontsize=13, fontweight='bold')

# ACF
plot_acf(
    serie_estacionaria,
    lags=50,
    ax=axes[0],
    alpha=0.05,
    title='ACF - Autocorrelación\n(Identifica orden MA: q)',
    color=COLORS['real']
)
axes[0].axvline(x=7, color='orange', linestyle='--',
                alpha=0.7, label='Lag estacional (7)')
axes[0].axvline(x=14, color='orange', linestyle='--', alpha=0.5)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PACF
plot_pacf(
    serie_estacionaria,
    lags=50,
    ax=axes[1],
    alpha=0.05,
    method='ywm',
    title='PACF - Autocorrelación Parcial\n(Identifica orden AR: p)',
    color=COLORS['sarima']
)
axes[1].axvline(x=7, color='orange', linestyle='--',
                alpha=0.7, label='Lag estacional (7)')
axes[1].axvline(x=14, color='orange', linestyle='--', alpha=0.5)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

print("📌 Interpretación:")
print("   → Picos significativos en ACF lag 1, 7 → sugieren MA(1) y S=7")
print("   → Picos significativos en PACF lag 1, 7 → sugieren AR(1) y estacionalidad")
print("   → Parámetros sugeridos: SARIMA(1,1,1)(1,1,1)[7]")


### DIVISIÓN TRAIN/TEST (Time Series Split)

TEORÍA: 

En series de tiempo NO se puede usar división aleatoria. El orden temporal es fundamental. Siempre: train = pasado, test = futuro. Usamos los últimos 90 días como conjunto de prueba.

In [ ]:

# Serie principal de trabajo
serie = df['valor']

# División temporal
n_test = 90  # Últimos 90 días para test

train = serie.iloc[:-n_test]
test = serie.iloc[-n_test:]

print(f"📊 División Train/Test:")
print(f"   → Train: {train.index[0].date()} a {train.index[-1].date()} ({len(train):,} obs)")
print(f"   → Test : {test.index[0].date()} a {test.index[-1].date()} ({len(test):,} obs)")
print(f"   → Proporción Test: {n_test/len(serie)*100:.1f}%")

# Visualización de la división
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=train.index, y=train,
    mode='lines', name='Entrenamiento',
    line=dict(color=COLORS['real'], width=1.5),
    fill='tozeroy', fillcolor='rgba(46, 134, 171, 0.1)'
))

fig.add_trace(go.Scatter(
    x=test.index, y=test,
    mode='lines', name='Prueba (Ground Truth)',
    line=dict(color=COLORS['test'], width=2)
))

fig.add_vline(
    x=test.index[0],
    line_dash="dash", line_color="red", line_width=2,
    annotation_text="🔪 Corte Train/Test"
)

fig.update_layout(
    title='División Temporal: Entrenamiento vs Prueba',
    xaxis_title='Fecha',
    yaxis_title='Valor',
    template='plotly_white',
    height=400
)
fig.show()

## Modelos

### 🤖 Implementación Comparativa de Modelos

#### Estrategia de Modelado
Implementaremos 4 modelos de diferente naturaleza:
| Modelo | Tipo | Fortaleza Principal |
|--------|------|---------------------|
| SARIMA | Estadístico Clásico | Interpretabilidad, datos pequeños |
| Prophet | Automatizado | Estacionalidades múltiples, robustez |
| XGBoost | Machine Learning | Captura no-linealidades |
| LSTM | Deep Learning | Patrones complejos de largo plazo |

### MODELO 1 - SARIMA

TEORÍA: 

SARIMA(p,d,q)(P,D,Q)[S]
  - (p,d,q): Componentes no estacionales (AR, Integración, MA)
  - (P,D,Q): Componentes estacionales
  - [S]: Período estacional

Los parámetros se seleccionaron basándose en los correlogramas ACF/PACF:
  p=1 (PACF corte en lag 1), d=1 (diferenciación), q=1 (ACF corte en lag 1)
  P=1, D=1, Q=1, S=7 (ciclo semanal identificado)

In [ ]:


print("⏳ Ajustando modelo SARIMA(1,1,1)(1,1,1)[7]...")
print("   (Este proceso puede tomar 1-2 minutos...)")

try:
    modelo_sarima = SARIMAX(
        train,
        order=(1, 1, 1),              # Parámetros no estacionales
        seasonal_order=(1, 1, 1, 7),  # Parámetros estacionales
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    resultado_sarima = modelo_sarima.fit(
        disp=False,     # Suprime output de optimización
        method='lbfgs'  # Método de optimización L-BFGS
    )

    # Pronóstico en-muestra y fuera-de-muestra
    pred_sarima = resultado_sarima.get_forecast(steps=n_test)
    pronostico_sarima = pred_sarima.predicted_mean
    intervalo_sarima = pred_sarima.conf_int(alpha=0.05)

    print(f"\n✅ SARIMA ajustado correctamente")
    print(f"   → AIC: {resultado_sarima.aic:.2f}")
    print(f"   → BIC: {resultado_sarima.bic:.2f}")
    print(resultado_sarima.summary().tables[1])

except Exception as e:
    print(f"⚠️  Error en SARIMA: {e}")
    print("    Usando parámetros simplificados ARIMA(1,1,1)...")
    modelo_sarima = SARIMAX(train, order=(1, 1, 1))
    resultado_sarima = modelo_sarima.fit(disp=False)
    pred_sarima = resultado_sarima.get_forecast(steps=n_test)
    pronostico_sarima = pred_sarima.predicted_mean
    intervalo_sarima = pred_sarima.conf_int(alpha=0.05)

### MODELO 2 - FACEBOOK PROPHET

TEORÍA: Prophet modela la serie como:
  y(t) = g(t) + s(t) + h(t) + εt

  g(t): Función de tendencia (lineal o logística con changepoints)
  s(t): Estacionalidades (Fourier series)
  h(t): Efectos de días festivos
  εt : Ruido

Ventaja: Detecta automáticamente changepoints (rupturas), lo cual es ideal para nuestra serie sintética con ruptura.

In [ ]:


print("⏳ Ajustando modelo Facebook Prophet...")

# Prophet requiere columnas 'ds' (fechas) y 'y' (valores)
df_prophet_train = pd.DataFrame({
    'ds': train.index,
    'y': train.values
})

# Configuración del modelo Prophet
modelo_prophet = Prophet(
    changepoint_prior_scale=0.3,    # Flexibilidad para detectar rupturas
    seasonality_mode='multiplicative',  # Consistente con nuestra generación
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95             # Intervalo de confianza al 95%
)

# Ajustar el modelo
modelo_prophet.fit(df_prophet_train, verbose=False)

# Crear dataframe futuro para pronóstico
futuro_prophet = modelo_prophet.make_future_dataframe(
    periods=n_test,
    freq='D'
)

# Generar pronóstico
forecast_prophet = modelo_prophet.predict(futuro_prophet)

# Extraer solo las predicciones del período de test
pronostico_prophet = pd.Series(
    forecast_prophet['yhat'].values[-n_test:],
    index=test.index
)

print("✅ Prophet ajustado correctamente")
print(f"   → Changepoints detectados: {len(modelo_prophet.changepoints)}")
print(f"   → Estacionalidades activas: anual, semanal")

# Visualización de changepoints detectados por Prophet
fig = modelo_prophet.plot(forecast_prophet)
fig.suptitle('Prophet: Pronóstico con Changepoints Detectados',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('prophet_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

# Componentes de Prophet
fig2 = modelo_prophet.plot_components(forecast_prophet)
fig2.suptitle('Prophet: Descomposición de Componentes',
              fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### MODELO 3 - XGBOOST CON VENTANAS DESLIZANTES (LAGS)

TEORÍA: 

XGBoost no entiende el tiempo directamente. Lo transformamos en un problema de regresión supervisada usando la técnica de LAGS:

  y(t) = f(y(t-1), y(t-2), ..., y(t-p), dia_semana, mes, ...)

Esto se llama "feature engineering" para series de tiempo. El número de lags (ventana) es un hiperparámetro clave.

In [ ]:

def crear_features_temporales(df_series, lags=14, incluir_fechas=True):
    """
    Transforma una serie de tiempo en features para modelos ML.

    Args:
        df_series: Serie pandas con índice datetime
        lags: Número de valores pasados a incluir como features
        incluir_fechas: Si incluir características de calendario

    Returns:
        DataFrame con features y target
    """
    df_feat = pd.DataFrame({'y': df_series})

    # === Features de Lag (Ventanas Deslizantes) ===
    for lag in range(1, lags + 1):
        df_feat[f'lag_{lag}'] = df_feat['y'].shift(lag)

    # === Features de Medias Móviles ===
    df_feat['ma_7'] = df_feat['y'].shift(1).rolling(window=7).mean()
    df_feat['ma_14'] = df_feat['y'].shift(1).rolling(window=14).mean()
    df_feat['std_7'] = df_feat['y'].shift(1).rolling(window=7).std()

    # === Features de Calendario ===
    if incluir_fechas:
        df_feat['dia_semana'] = df_series.index.dayofweek
        df_feat['mes'] = df_series.index.month
        df_feat['dia_anio'] = df_series.index.dayofyear
        df_feat['semana'] = df_series.index.isocalendar().week.astype(int)
        df_feat['trimestre'] = df_series.index.quarter

    df_feat.dropna(inplace=True)
    return df_feat

# Crear features para toda la serie
n_lags = 14
df_features = crear_features_temporales(serie, lags=n_lags)

X = df_features.drop('y', axis=1)
y_target = df_features['y']

# División respetando el orden temporal
X_train = X.iloc[:-n_test]
X_test = X.iloc[-n_test:]
y_train_xgb = y_target.iloc[:-n_test]
y_test_xgb = y_target.iloc[-n_test:]

print(f"📊 Features XGBoost:")
print(f"   → Total features: {X.shape[1]}")
print(f"   → Features: {list(X.columns)}")

# === Entrenamiento XGBoost ===
print("\n⏳ Entrenando XGBoost...")

modelo_xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbosity=0
)

modelo_xgb.fit(
    X_train, y_train_xgb,
    eval_set=[(X_test, y_test_xgb)],
    verbose=False
)

pronostico_xgb = pd.Series(
    modelo_xgb.predict(X_test),
    index=test.index[-len(X_test):]
)

print("✅ XGBoost entrenado correctamente")

# Importancia de Features
feat_importance = pd.Series(
    modelo_xgb.feature_importances_,
    index=X.columns
).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
feat_importance.plot(kind='barh', ax=ax, color=COLORS['xgboost'], alpha=0.8)
ax.set_title('XGBoost: Importancia de Features', fontsize=13, fontweight='bold')
ax.set_xlabel('Importancia (F-Score)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('xgboost_features.png', dpi=150, bbox_inches='tight')
plt.show()



### MODELO 4 - LSTM (Deep Learning)

TEORÍA: 
Las LSTM (Long Short-Term Memory) son redes neuronales recurrentes diseñadas para capturar dependencias de largo plazo en secuencias.

Arquitectura: [Input] → [LSTM] → [Dropout] → [LSTM] → [Dense] → [Output]

La técnica de ventanas deslizantes convierte la serie en muestras 3D: (n_muestras, n_pasos_tiempo, n_features)

In [ ]:


print("⏳ Preparando y entrenando modelo LSTM...")

# === Normalización ===
# Las LSTM funcionan mejor con datos en [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
serie_escalada = scaler.fit_transform(serie.values.reshape(-1, 1))

# === Crear Secuencias para LSTM ===
def crear_secuencias_lstm(datos, n_pasos):
    """
    Convierte serie 1D en secuencias 3D para LSTM.
    Input shape: (n, 1)
    Output shapes: X=(n-n_pasos, n_pasos, 1), y=(n-n_pasos, 1)
    """
    X_seq, y_seq = [], []
    for i in range(len(datos) - n_pasos):
        X_seq.append(datos[i:i + n_pasos])
        y_seq.append(datos[i + n_pasos])
    return np.array(X_seq), np.array(y_seq)

N_PASOS = 14  # Ventana de contexto: 14 días anteriores

X_seq, y_seq = crear_secuencias_lstm(serie_escalada, N_PASOS)

# División temporal (mantenemos n_test para comparación)
X_train_lstm = X_seq[:-n_test]
X_test_lstm = X_seq[-n_test:]
y_train_lstm = y_seq[:-n_test]
y_test_lstm = y_seq[-n_test:]

print(f"   → Shape X_train: {X_train_lstm.shape}")
print(f"   → Shape X_test : {X_test_lstm.shape}")

# === Arquitectura LSTM ===
tf.random.set_seed(42)

modelo_lstm = Sequential([
    # Primera capa LSTM con return_sequences para apilar LSTMs
    LSTM(64, return_sequences=True,
         input_shape=(N_PASOS, 1),
         name='lstm_1'),
    Dropout(0.2, name='dropout_1'),

    # Segunda capa LSTM
    LSTM(32, return_sequences=False,
         name='lstm_2'),
    Dropout(0.2, name='dropout_2'),

    # Capa de salida
    Dense(16, activation='relu', name='dense_1'),
    Dense(1, name='output')
], name='LSTM_Forecaster')

modelo_lstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='huber',    # Huber loss: más robusto a outliers que MSE
    metrics=['mae']
)

print("\n📐 Arquitectura del modelo LSTM:")
modelo_lstm.summary()

# === Entrenamiento ===
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=0
)

historia_lstm = modelo_lstm.fit(
    X_train_lstm, y_train_lstm,
    epochs=100,
    batch_size=32,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=0
)

print(f"\n✅ LSTM entrenado ({len(historia_lstm.history['loss'])} epochs)")

# === Pronóstico ===
pred_lstm_escalada = modelo_lstm.predict(X_test_lstm, verbose=0)
pred_lstm_original = scaler.inverse_transform(pred_lstm_escalada)

pronostico_lstm = pd.Series(
    pred_lstm_original.flatten(),
    index=test.index[-len(pred_lstm_original):]
)

# Curva de aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(historia_lstm.history['loss'], label='Train Loss',
             color=COLORS['lstm'], linewidth=2)
axes[0].plot(historia_lstm.history['val_loss'], label='Val Loss',
             color=COLORS['xgboost'], linewidth=2, linestyle='--')
axes[0].set_title('LSTM: Curva de Aprendizaje (Loss)', fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Huber Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(historia_lstm.history['mae'], label='Train MAE',
             color=COLORS['lstm'], linewidth=2)
axes[1].plot(historia_lstm.history['val_mae'], label='Val MAE',
             color=COLORS['xgboost'], linewidth=2, linestyle='--')
axes[1].set_title('LSTM: Curva de Aprendizaje (MAE)', fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lstm_learning.png', dpi=150, bbox_inches='tight')
plt.show()


## EVALUACIÓN Y MÉTRICAS COMPARATIVAS

TEORÍA: 

Métricas de evaluación para pronósticos:

  MAE  (Mean Absolute Error):
    → Promedio de errores absolutos. Interpretación directa en unidades.

  RMSE (Root Mean Squared Error):
    → Penaliza más los errores grandes. Sensible a outliers.

  MAPE (Mean Absolute Percentage Error):
    → Error relativo en %. Útil para comparar series de diferente escala.
    → MAPE < 10% → Excelente | 10-20% → Bueno | >20% → Revisar

Regla de oro: Siempre usar MÚLTIPLES métricas, nunca una sola.

In [ ]:

def calcular_metricas(real, predicho, nombre_modelo):
    """
    Calcula MAE, RMSE y MAPE para un conjunto de predicciones.
    """
    # Alinear longitudes
    min_len = min(len(real), len(predicho))
    real_alin = real.values[-min_len:]
    pred_alin = predicho.values[-min_len:]

    mae = mean_absolute_error(real_alin, pred_alin)
    rmse = np.sqrt(mean_squared_error(real_alin, pred_alin))
    mape = np.mean(np.abs((real_alin - pred_alin) / real_alin)) * 100

    return {
        'Modelo': nombre_modelo,
        'MAE': round(mae, 4),
        'RMSE': round(rmse, 4),
        'MAPE (%)': round(mape, 4)
    }

# Calcular métricas para cada modelo
metricas = [
    calcular_metricas(test, pronostico_sarima, 'SARIMA(1,1,1)(1,1,1)[7]'),
    calcular_metricas(test, pronostico_prophet, 'Facebook Prophet'),
    calcular_metricas(test, pronostico_xgb, 'XGBoost (Lags)'),
    calcular_metricas(test, pronostico_lstm, 'LSTM')
]

df_metricas = pd.DataFrame(metricas).set_index('Modelo')

# Identificar el mejor modelo por RMSE
mejor_modelo = df_metricas['RMSE'].idxmin()

print("=" * 60)
print("  📊 TABLA COMPARATIVA DE MÉTRICAS DE ERROR")
print("=" * 60)
print(df_metricas.to_string())
print(f"\n🏆 MEJOR MODELO (menor RMSE): {mejor_modelo}")
print(f"   RMSE = {df_metricas.loc[mejor_modelo, 'RMSE']:.4f}")
print(f"   MAE  = {df_metricas.loc[mejor_modelo, 'MAE']:.4f}")
print(f"   MAPE = {df_metricas.loc[mejor_modelo, 'MAPE (%)']:.2f}%")

# === Visualización de Métricas ===
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Comparativa de Métricas por Modelo', fontsize=13, fontweight='bold')

colores_modelos = [COLORS['sarima'], COLORS['prophet'],
                   COLORS['xgboost'], COLORS['lstm']]
metricas_plot = ['MAE', 'RMSE', 'MAPE (%)']

for ax, metrica in zip(axes, metricas_plot):
    bars = ax.bar(
        df_metricas.index,
        df_metricas[metrica],
        color=colores_modelos,
        alpha=0.85,
        edgecolor='black',
        linewidth=0.5
    )
    ax.set_title(metrica, fontsize=11, fontweight='bold')
    ax.set_xticklabels(df_metricas.index, rotation=30, ha='right', fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

    # Destacar el mejor
    idx_min = df_metricas[metrica].argmin()
    bars[idx_min].set_edgecolor('gold')
    bars[idx_min].set_linewidth(3)

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height * 1.01,
                f'{height:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('metricas_comparativa.png', dpi=150, bbox_inches='tight')
plt.show()


### GRÁFICO FINAL - PRONÓSTICOS SUPERPUESTOS

El gráfico final es la visualización más importante del análisis:
superpone todos los pronósticos contra los datos reales para evaluar visualmente el comportamiento de cada modelo.

In [ ]:

# Determinar la longitud mínima para alinear todos los pronósticos
min_len_pred = min(
    len(pronostico_sarima), len(pronostico_prophet),
    len(pronostico_xgb), len(pronostico_lstm)
)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        '🤖 Pronóstico de Todos los Modelos vs Datos Reales',
        '📊 Error Absoluto por Modelo'
    ],
    row_heights=[0.7, 0.3],
    vertical_spacing=0.12
)

# === Panel Superior: Contexto histórico + pronósticos ===
# Últimos 30 días de entrenamiento para contexto
contexto = train.iloc[-30:]

# Datos históricos (contexto)
fig.add_trace(go.Scatter(
    x=contexto.index, y=contexto,
    mode='lines', name='Histórico',
    line=dict(color='lightgray', width=1.5),
    showlegend=True
), row=1, col=1)

# Datos reales de test
fig.add_trace(go.Scatter(
    x=test.index[-min_len_pred:], y=test.values[-min_len_pred:],
    mode='lines+markers', name='✅ Real (Test)',
    line=dict(color=COLORS['real'], width=2.5),
    marker=dict(size=3)
), row=1, col=1)

# Pronóstico SARIMA con intervalo de confianza
fig.add_trace(go.Scatter(
    x=pronostico_sarima.index[-min_len_pred:],
    y=pronostico_sarima.values[-min_len_pred:],
    mode='lines', name='SARIMA',
    line=dict(color=COLORS['sarima'], width=2, dash='dash')
), row=1, col=1)

# Intervalo de confianza SARIMA
fig.add_trace(go.Scatter(
    x=pd.concat([
        pd.Series(intervalo_sarima.iloc[-min_len_pred:, 0].index),
        pd.Series(intervalo_sarima.iloc[-min_len_pred:, 1].index[::-1])
    ]),
    y=np.concatenate([
        intervalo_sarima.iloc[-min_len_pred:, 0].values,
        intervalo_sarima.iloc[-min_len_pred:, 1].values[::-1]
    ]),
    fill='toself',
    fillcolor='rgba(162, 59, 114, 0.1)',
    line=dict(color='rgba(255,255,255,0)'),
    name='IC SARIMA 95%',
    showlegend=True
), row=1, col=1)

# Prophet
fig.add_trace(go.Scatter(
    x=pronostico_prophet.index[-min_len_pred:],
    y=pronostico_prophet.values[-min_len_pred:],
    mode='lines', name='Prophet',
    line=dict(color=COLORS['prophet'], width=2, dash='dot')
), row=1, col=1)

# XGBoost
fig.add_trace(go.Scatter(
    x=pronostico_xgb.index[-min_len_pred:],
    y=pronostico_xgb.values[-min_len_pred:],
    mode='lines', name='XGBoost',
    line=dict(color=COLORS['xgboost'], width=2, dash='dashdot')
), row=1, col=1)

# LSTM
fig.add_trace(go.Scatter(
    x=pronostico_lstm.index[-min_len_pred:],
    y=pronostico_lstm.values[-min_len_pred:],
    mode='lines', name='LSTM',
    line=dict(color=COLORS['lstm'], width=2)
), row=1, col=1)

# === Panel Inferior: Error Absoluto ===
real_alin = test.values[-min_len_pred:]
idx_alin = test.index[-min_len_pred:]

errores = {
    'SARIMA': np.abs(real_alin - pronostico_sarima.values[-min_len_pred:]),
    'Prophet': np.abs(real_alin - pronostico_prophet.values[-min_len_pred:]),
    'XGBoost': np.abs(real_alin - pronostico_xgb.values[-min_len_pred:]),
    'LSTM': np.abs(real_alin - pronostico_lstm.values[-min_len_pred:])
}

colores_err = [COLORS['sarima'], COLORS['prophet'],
               COLORS['xgboost'], COLORS['lstm']]

for (nombre, error_vals), color in zip(errores.items(), colores_err):
    fig.add_trace(go.Scatter(
        x=idx_alin, y=error_vals,
        mode='lines', name=f'Error |{nombre}|',
        line=dict(color=color, width=1.5),
        opacity=0.8,
        showlegend=False
    ), row=2, col=1)

# Layout
fig.update_layout(
    height=750,
    title_text='<b>📊 Comparativa Final de Modelos de Pronóstico</b>',
    title_font_size=16,
    template='plotly_white',
    legend=dict(
        orientation='v',
        x=1.02, y=0.9,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='gray',
        borderwidth=1
    ),
    hovermode='x unified'
)

fig.update_yaxes(title_text='Valor', row=1, col=1)
fig.update_yaxes(title_text='Error Abs.', row=2, col=1)
fig.update_xaxes(title_text='Fecha', row=2, col=1)

fig.show()
print("✅ Gráfico comparativo final generado")


### ANÁLISIS DE RESIDUOS DEL MEJOR MODELO

TEORÍA: 

Los residuos de un buen modelo deben cumplir:
  1. Normalidad: ε ~ N(0, σ²)
  2. Media cero: E[ε] ≈ 0
  3. Homocedasticidad: Varianza constante
  4. Ausencia de autocorrelación: Corr(εt, εt-k) ≈ 0 para k ≠ 0

Si los residuos tienen autocorrelación → El modelo no capturó toda la información y puede mejorarse.

In [ ]:


print(f"🔍 Analizando residuos del mejor modelo: {mejor_modelo}")

# Identificar el pronóstico del mejor modelo
pronósticos_dict = {
    'SARIMA(1,1,1)(1,1,1)[7]': pronostico_sarima,
    'Facebook Prophet': pronostico_prophet,
    'XGBoost (Lags)': pronostico_xgb,
    'LSTM': pronostico_lstm
}

mejor_pronostico = pronósticos_dict[mejor_modelo]
min_len_res = min(len(test), len(mejor_pronostico))

# Calcular residuos
residuos = pd.Series(
    test.values[-min_len_res:] - mejor_pronostico.values[-min_len_res:],
    index=test.index[-min_len_res:]
)

# === 1. Test de Normalidad (Jarque-Bera) ===
jb_stat, jb_pvalue = stats.jarque_bera(residuos)
shapiro_stat, shapiro_pvalue = stats.shapiro(residuos[:50])  # Shapiro para n<50

# === 2. Test de Ljung-Box (Autocorrelación de Residuos) ===
lb_resultado = acorr_ljungbox(residuos, lags=[10, 20], return_df=True)

print(f"\n{'='*55}")
print(f"  📋 Diagnóstico de Residuos - {mejor_modelo}")
print(f"{'='*55}")
print(f"\n  Estadísticas Descriptivas:")
print(f"  Media     : {residuos.mean():.4f} (ideal ≈ 0)")
print(f"  Desv. Std : {residuos.std():.4f}")
print(f"  Skewness  : {stats.skew(residuos):.4f} (ideal ≈ 0)")
print(f"  Kurtosis  : {stats.kurtosis(residuos):.4f} (ideal ≈ 0)")

print(f"\n  Test de Normalidad:")
print(f"  → Jarque-Bera: stat={jb_stat:.4f}, p={jb_pvalue:.4f}")
if jb_pvalue > 0.05:
    print(f"    ✅ Residuos NORMALES (p > 0.05)")
else:
    print(f"    ⚠️  Residuos no normales (p ≤ 0.05)")

print(f"\n  Test Ljung-Box (Autocorrelación Residual):")
print(lb_resultado[['lb_stat', 'lb_pvalue']].round(4))
if (lb_resultado['lb_pvalue'] > 0.05).all():
    print(f"    ✅ Sin autocorrelación significativa en residuos")
else:
    print(f"    ⚠️  Autocorrelación detectada → Modelo mejorable")

# === Visualización Diagnóstica ===
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(f'Análisis Diagnóstico de Residuos\nModelo: {mejor_modelo}',
             fontsize=13, fontweight='bold')

# 1. Serie de Residuos
axes[0, 0].plot(residuos, color=COLORS['real'],
                linewidth=1, alpha=0.8)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[0, 0].axhline(y=2*residuos.std(), color='orange',
                   linestyle=':', linewidth=1, label='±2σ')
axes[0, 0].axhline(y=-2*residuos.std(), color='orange',
                   linestyle=':', linewidth=1)
axes[0, 0].set_title('Residuos en el Tiempo', fontweight='bold')
axes[0, 0].set_ylabel('Residuo')
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True, alpha=0.3)

# 2. Distribución de Residuos
axes[0, 1].hist(residuos, bins=20, color=COLORS['sarima'],
                alpha=0.7, edgecolor='black', density=True)
x_norm = np.linspace(residuos.min(), residuos.max(), 100)
axes[0, 1].plot(x_norm,
                stats.norm.pdf(x_norm, residuos.mean(), residuos.std()),
                'r-', linewidth=2, label='Normal Teórica')
axes[0, 1].set_title('Distribución de Residuos', fontweight='bold')
axes[0, 1].set_xlabel('Residuo')
axes[0, 1].set_ylabel('Densidad')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Q-Q Plot
(osm, osr), (slope, intercept, r) = stats.probplot(residuos)
axes[0, 2].scatter(osm, osr, alpha=0.6, color=COLORS['xgboost'], s=20)
linea_qq = np.array([min(osm), max(osm)])
axes[0, 2].plot(linea_qq, slope * linea_qq + intercept,
                'r-', linewidth=2, label=f'R²={r**2:.4f}')
axes[0, 2].set_title('Q-Q Plot (Normalidad)', fontweight='bold')
axes[0, 2].set_xlabel('Cuantiles Teóricos')
axes[0, 2].set_ylabel('Cuantiles Observados')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# 4. ACF de Residuos
plot_acf(residuos, lags=30, ax=axes[1, 0],
         alpha=0.05, title='ACF de Residuos\n(Debe estar dentro de bandas)',
         color=COLORS['real'])
axes[1, 0].grid(True, alpha=0.3)

# 5. PACF de Residuos
plot_pacf(residuos, lags=30, ax=axes[1, 1],
          alpha=0.05, method='ywm',
          title='PACF de Residuos\n(Debe estar dentro de bandas)',
          color=COLORS['sarima'])
axes[1, 1].grid(True, alpha=0.3)

# 6. Residuos vs Valores Ajustados
valores_ajustados = mejor_pronostico.values[-min_len_res:]
axes[1, 2].scatter(valores_ajustados, residuos,
                   alpha=0.6, color=COLORS['prophet'], s=20)
axes[1, 2].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
z = np.polyfit(valores_ajustados, residuos, 1)
p = np.poly1d(z)
axes[1, 2].plot(np.sort(valores_ajustados),
                p(np.sort(valores_ajustados)),
                "b--", alpha=0.7, linewidth=1.5, label='Tendencia')
axes[1, 2].set_title('Residuos vs Ajustados\n(Homocedasticidad)',
                     fontweight='bold')
axes[1, 2].set_xlabel('Valores Ajustados')
axes[1, 2].set_ylabel('Residuos')
axes[1, 2].legend(fontsize=8)
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('analisis_residuos.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Análisis de residuos completado")



## RESUMEN EJECUTIVO FINAL

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# CELDA 18: RESUMEN EJECUTIVO FINAL
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*65)
print("  🏆  RESUMEN EJECUTIVO - LABORATORIO DE SERIES DE TIEMPO")
print("="*65)

print("\n📊 TABLA FINAL DE MÉTRICAS:")
print("-"*55)

# Tabla con formato
df_reporte = df_metricas.copy()
df_reporte['Ranking (RMSE)'] = df_reporte['RMSE'].rank().astype(int)
df_reporte = df_reporte.sort_values('RMSE')

for idx, (modelo, row) in enumerate(df_reporte.iterrows()):
    emoji = "🥇" if idx == 0 else "🥈" if idx == 1 else "🥉" if idx == 2 else "4️⃣"
    print(f"\n  {emoji} {modelo}")
    print(f"      MAE  = {row['MAE']:.4f}")
    print(f"      RMSE = {row['RMSE']:.4f}")
    print(f"      MAPE = {row['MAPE (%)']:.2f}%")

print(f"\n{'='*65}")
print(f"  🎯 MODELO RECOMENDADO: {mejor_modelo}")
print(f"{'='*65}")

print("""
📝 LECCIONES APRENDIDAS:
   1. El EDA es fundamental → Guía la selección del modelo correcto
   2. SARIMA: Ideal para series con patrones claros y datos limitados
   3. Prophet: Robusto ante rupturas, estacionalidades múltiples y fácil uso
   4. XGBoost: Poderoso cuando el feature engineering es bien diseñado
   5. LSTM: Potente para secuencias largas, pero requiere más datos y tuning
   
⚠️  ADVERTENCIAS IMPORTANTES:
   • Nunca uses datos de test para ajustar el modelo (data leakage)
   • Una métrica alta no garantiza buenas predicciones fuera del período
   • Siempre analiza los residuos, no sólo las métricas de error
   • En producción: reentrenar periódicamente y monitorear deriva (drift)

🔄 SIGUIENTES PASOS:
   • Implementar validación cruzada temporal (TimeSeriesSplit)
   • Explorar modelos ensemble (combinación de pronósticos)
   • Incorporar variables exógenas (SARIMAX, Prophet regressors)
   • Implementar detección automática de changepoints
""")

print("✅ Laboratorio completado exitosamente")
print("   Archivos generados: descomposicion_serie.png, estacionariedad.png,")
print("   acf_pacf.png, prophet_forecast.png, xgboost_features.png,")
print("   lstm_learning.png, metricas_comparativa.png, analisis_residuos.png")